# p109 — weights move, the representation holds

Two companion notebooks ([late reorganization](./p109_late_reorganization.ipynb), [blow-up / slingshot](./p109_blowup_slingshot.ipynb)) established that the ~27k event is a *functional* round-trip: the parameters drift and a few neurons explode, but the function returns to the same place. This notebook widens that lens to the whole of training and asks the structural question underneath it:

**Where does the change live — in the representation, or in the weights?**

The hypothesis the data suggests: the **representation forms early** (during memorization), is then **held nearly fixed** while the **weights keep moving**, with a brief disorganizing event *during grokking* and a milder one at ~27k. If true, most of the late weight motion is *gauge* — rotation that doesn't change what the network computes.

The representation question needs a **gauge-invariant** comparison. Raw activation cosine would be polluted by exactly the rotation we suspect dominates; **linear CKA** measures representational similarity up to any linear transform, so it isolates whether the *representation* changed, independent of the frame the weights express it in. (This is the activation-space analog of notebook 1's gauge-blind centroid geometry.)

For the weight side we borrow **Kosson, Messmer & Jaggi (2023), *Rotational Equilibrium: How Weight Decay Balances Learning Across Neural Networks***. Their result: with weight decay, each neuron's weight vector reaches an equilibrium where the shrink from decay balances the growth from the gradient — the norm stabilizes and the update becomes a near-constant **angular rotation** on a fixed-radius sphere. Weight decay's real job is to set the *rotational* learning rate. This gives a precise vocabulary for "weights move without changing the function": on the plateau the weights should sit at **rotational equilibrium** (stable norm, ongoing rotation), and the ~27k exploders should be neurons that *break* it with a radial escape.

**Plan:** §1 measures cross-site representation similarity (CKA) across training; §2 measures the weight rotational dynamics (Kosson decomposition) on the same epochs; the reading puts them on one axis.

---
*Conventions follow the other p109 notebooks: all access through the miscope API; the `=` query (position 2) is the readout site; CKA is computed on the full (a,b) grid. `wd = 1.0` for this family — the weight decay whose rotational role §2 reads out.*

In [ ]:
import os
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
from miscope.families.discovery import load_family_from_dir

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
variant = fam.get_variant(prime=109, seed=485, data_seed=598)
PRIME = 109
GRID = [[a, b] for a in range(PRIME) for b in range(PRIME)]
GROK = (2000, 5000)        # grokking transition window
EVENT = (26900, 30000)     # the ~27k destabilize-and-recover window


def linear_cka(X, Y):
    # Gauge-invariant representational similarity in [0,1] (Kornblith et al. 2019).
    X = X - X.mean(0); Y = Y - Y.mean(0)
    num = np.linalg.norm(Y.T @ X, "fro") ** 2
    den = np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")
    return float(num / den)


variant

## 1. Where the representation lives — CKA across sites and training

Linear CKA between a checkpoint's activations and a **reference** checkpoint answers "is the representation already in its final form?" — without caring how the weights rotated to express it. We read it at four sites along the computation, all at the `=` query: the **operand embedding** (`embed`, position 0 — how the *numbers* are represented), the **attention output**, the **MLP hidden** activations, and the **resid_post** (the vector that gets unembedded). Reference = the final checkpoint; high CKA early = the representation formed early.

A drop in CKA-to-final that later recovers is a **disorganizing event**: the representation churning and re-cohering. We expect one at the grokking transition and, from the companion notebooks, a milder one at ~27k.

In [ ]:
# Per-site = -query representation at each epoch (one forward pass per checkpoint).
SITES = {"embed_a": ("embed.hook_out", 0), "attn_out": ("blocks.0.attn.hook_out", 2),
         "mlp_hidden": ("blocks.0.mlp.hook_out", 2), "resid_post": ("blocks.0.hook_out", 2)}
COLORS = {"embed_a": "#9467bd", "attn_out": "#d62728", "mlp_hidden": "#2ca02c", "resid_post": "#1f77b4"}


def site_reps(epoch):
    _, cache = variant.run_with_cache(variant.make_probe(GRID), epoch=int(epoch))
    return {n: cache[k][:, pos, :].detach().cpu().numpy().astype(np.float64) for n, (k, pos) in SITES.items()}


all_ep = np.array(variant.artifacts.get_epochs("parameter_snapshot"))
want = [0, 200, 500, 1000, 1500, 2000, 2500, 3000, 4000, 5000, 7000, 10000, 20000,
        26000, 27400, 27600, 28500, 30000, int(all_ep[-1])]
cka_ep = sorted({int(all_ep[np.argmin(np.abs(all_ep - w))]) for w in want})
REPS = {e: site_reps(e) for e in cka_ep}
ref = cka_ep[-1]
cka = {s: np.array([linear_cka(REPS[e][s], REPS[ref][s]) for e in cka_ep]) for s in SITES}
print(f"computed CKA-to-final ({ref}) at {len(cka_ep)} epochs, {cka_ep[0]}..{cka_ep[-1]}")

In [ ]:
fig = go.Figure()
fig.add_vrect(x0=GROK[0], x1=GROK[1], fillcolor="green", opacity=0.07, line_width=0,
              annotation_text="grokking", annotation_position="top left")
fig.add_vrect(x0=EVENT[0], x1=EVENT[1], fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top right")
for s in SITES:
    fig.add_trace(go.Scatter(x=cka_ep, y=cka[s], mode="lines+markers", name=s,
                             line=dict(color=COLORS[s])))
fig.update_layout(title="§1 — representational similarity to the final checkpoint (linear CKA, gauge-invariant)",
                  xaxis_title="epoch", yaxis_title="CKA to final", yaxis_range=[0, 1.02],
                  legend=dict(x=0.99, y=0.01, xanchor="right", yanchor="bottom"), height=460)
fig.show()

ce = np.array(cka_ep)
for s in SITES:
    formed = ce[np.argmax(cka[s] >= 0.9)] if (cka[s] >= 0.9).any() else None
    i3k = int(np.argmin(np.abs(ce - 3000)))
    print(f"{s:>11}: CKA@3000={cka[s][i3k]:.2f}  reaches 0.90 by epoch {formed}  "
          f"plateau(>=10k) min {cka[s][ce >= 10000].min():.3f}")
ig = (ce >= 3000) & (ce <= 5000)
print("\ngrokking-window CKA-to-final (look for a dip at ~4000 then recovery):")
for e in ce[ig]:
    i = int(np.where(ce == e)[0][0])
    print(f"  epoch {e:>5}: " + "  ".join(f"{s} {cka[s][i]:.3f}" for s in SITES))

### Adapting the round-trip box-plot to the representation

§3 of the blow-up notebook boxed the per-**input** logit *margin* across the event to show the output round-trip. Here is the representation-level analog: for each of the p² inputs, the cosine of its `resid_post` vector to the **pre-event reference (26000)**, boxed per epoch across the event. Because the gauge barely moves across the event (CKA ≈ 1 throughout the window, §1), this cosine is meaningful — it shows the per-input representation *dip and return*, the structural face of the functional round-trip.

In [ ]:
REF_EVENT = 26000
event_ep = sorted({int(all_ep[np.argmin(np.abs(all_ep - e))])
                   for e in [26000, 26900, 27400, 27600, 28500, 29000, 30000, 34999]})
R_ref = (REPS.get(REF_EVENT) or site_reps(REF_EVENT))["resid_post"]


def per_input_cos(R, Rref):
    return (R * Rref).sum(1) / (np.linalg.norm(R, axis=1) * np.linalg.norm(Rref, axis=1) + 1e-12)


fig = go.Figure()
for e in event_ep:
    R = (REPS.get(e) or site_reps(e))["resid_post"]
    fig.add_trace(go.Box(y=per_input_cos(R, R_ref), name=str(e), boxpoints=False))
fig.update_layout(title="Representation round-trip: per-input resid_post cosine to pre-event (26000)",
                  xaxis_title="epoch", yaxis_title="cosine to epoch-26000 representation",
                  showlegend=False, height=440)
fig.show()
med = {e: float(np.median(per_input_cos((REPS.get(e) or site_reps(e))["resid_post"], R_ref))) for e in event_ep}
worst = min(med, key=med.get)
print(f"most-perturbed checkpoint: {worst} (median per-input cosine {med[worst]:.4f}); "
      f"recovers by {event_ep[-1]} to {med[event_ep[-1]]:.4f}")

## 2. Where the weight motion goes — rotational equilibrium (Kosson)

If the representation is fixed on the plateau but the weights keep moving, the motion must be **gauge**: rotation in directions the function is blind to. Kosson's decomposition makes this measurable. For each neuron's `W_in` row, the step `Δw = w_{t+1} − w_t` splits into a **radial** part (change of norm, along `ŵ`) and a **tangential** part (rotation, perpendicular to `w`), and the **angular update** is the angle swept per step.

Rotational equilibrium predicts, on the plateau: **stable norm** (radial ≈ 0), **ongoing rotation** (angular update > 0), motion dominated by the **tangential** component. The ~27k exploders should be the exception — a **radial escape** that breaks equilibrium.

In [ ]:
# Per-neuron W_in trajectories and the radial/tangential decomposition of each step.
ps = np.array(variant.artifacts.get_epochs("parameter_snapshot"))
W = np.stack([variant.artifacts.load_epoch("parameter_snapshot", int(e))["W_in"].T.astype(np.float64) for e in ps])
norm = np.linalg.norm(W, axis=2)                       # (E, N)
w0, w1 = W[:-1], W[1:]
n0 = np.linalg.norm(w0, axis=2) + 1e-12; n1 = np.linalg.norm(w1, axis=2) + 1e-12
ang = np.degrees(np.arccos(np.clip((w0 * w1).sum(2) / (n0 * n1), -1, 1)))   # (E-1, N) angular update
dw = w1 - w0
radial = (dw * (w0 / n0[:, :, None])).sum(2)                                # (E-1, N) signed norm change
tang = np.sqrt(np.clip((dw ** 2).sum(2) - radial ** 2, 0, None))           # (E-1, N) rotation magnitude
mid = ps[:-1]

fig = go.Figure()
fig.add_vrect(x0=GROK[0], x1=GROK[1], fillcolor="green", opacity=0.07, line_width=0,
              annotation_text="grokking", annotation_position="top left")
fig.add_vrect(x0=EVENT[0], x1=EVENT[1], fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top right")
fig.add_trace(go.Scatter(x=mid, y=np.median(ang, 1), mode="lines", name="median angular update / step",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=ps, y=np.median(norm, 1), mode="lines", name="median ||W_in[j]|| (norm)",
                         line=dict(color="#2ca02c"), yaxis="y2"))
fig.update_layout(title="§2 — rotation continues at stable norm: plateau = rotational equilibrium",
                  xaxis_title="epoch", yaxis_title="angular update (deg/step)",
                  yaxis2=dict(title="median ||W_in[j]||", overlaying="y", side="right"),
                  legend=dict(x=0.99, y=0.99, xanchor="right"), height=460)
fig.show()

def win(lo, hi): return (mid >= lo) & (mid <= hi)
for nm, m in [("grokking 2-5k", win(*GROK)), ("plateau 8-26k", win(8000, 26000))]:
    tt = np.median(tang[m] / (np.sqrt(radial[m] ** 2 + tang[m] ** 2) + 1e-12))
    print(f"{nm:>14}: angular {np.median(ang[m]):.3f} deg/step | median |radial| {np.median(np.abs(radial[m])):.4f} "
          f"| tangential fraction {tt:.2f}  (rotation at ~constant norm => equilibrium)")

In [ ]:
# The exploders break rotational equilibrium with a radial escape.
ev = win(26500, 29000)
i20, i285 = int(np.argmin(np.abs(ps - 20000))), int(np.argmin(np.abs(ps - 28500)))
fig = go.Figure()
fig.add_vrect(x0=EVENT[0], x1=EVENT[1], fillcolor="orange", opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=mid, y=np.median(np.abs(radial), 1), mode="lines",
                         name="population median |radial|", line=dict(color="gray")))
for j, c in [(327, "#d62728"), (400, "#ff7f0e")]:
    fig.add_trace(go.Scatter(x=mid, y=np.abs(radial[:, j]), mode="lines", name=f"|radial| n{j}", line=dict(color=c)))
fig.update_layout(title="Exploders break equilibrium: radial (norm-changing) escape at the event",
                  xaxis_title="epoch", xaxis_range=[24000, 31000], yaxis_title="|radial component| / step",
                  legend=dict(x=0.01, y=0.99), height=440)
fig.show()
print(f"{'neuron':>7} {'|radial|/step max':>18} {'norm 20k->28.5k':>18} {'max angular(deg)':>17}")
print(f"{'pop':>7} {np.median(np.abs(radial[ev])):>18.4f} {'(median)':>18} {'-':>17}")
for j in (327, 400):
    print(f"{j:>7} {np.abs(radial[ev, j]).max():>18.3f} {f'{norm[i20, j]:.2f} -> {norm[i285, j]:.2f}':>18} {ang[ev, j].max():>17.1f}")
print("\nexploder |radial|/step is 1-2 orders above the population median: a radial escape, not rotation")

## Reading — weights move, the representation holds

Put the two axes together and the division of labor is clean.

**The representation forms during memorization, not at grokking's end.** CKA-to-final climbs steadily through epochs 0–3000 — by epoch 3000 the `=`-query representation is already ~80% of its final form at every site (attn 0.79, mlp 0.82, resid_post 0.77). Grokking does not *build* the representation; it **finishes and locks** it. By ~epoch 5000 CKA is ≥0.95 and it then holds essentially flat (0.94–1.0) all the way to 26k.

**There is a disorganizing event during grokking — at ~epoch 4000.** CKA-to-final *dips* right before it locks (resid_post 0.77 → 0.67 from 3000→4000, mlp 0.82 → 0.77), and the largest step-to-step representational change of the whole run sits in 3000→5000. The representation briefly churns and re-coheres — exactly the "possible disorganizing event during grokking" the data hinted at, now located.

**On the plateau the weights keep moving — by rotating at fixed norm.** While CKA sits at ~1.0 (5k–26k), the `W_in` rows are *not* still: the median neuron rotates ~**0.19°/step** with the motion **~83% tangential** and the radial (norm-changing) part near zero. That is rotational equilibrium in the Kosson sense — weight decay (`wd = 1.0`) balancing the gradient so the norm holds and the update is pure rotation. The late weight motion the companion notebooks saw is **gauge**: rotation in directions the representation is blind to, which is why CKA does not move.

**The ~27k event is a small set of neurons breaking equilibrium radially.** The exploders are not rotating faster — they **escape radially**: n327's `|radial|/step` jumps to ~0.62 (population ~0.008) and its norm runs 0.06 → 2.92, n400 similarly. This radial escape is what briefly perturbs the representation (the mild CKA dip at 27400–28500 and the per-input cosine trough), before relaxing back — the structural reading of the functional round-trip.

**Net.** Across all of p109's training the *representation* changes in exactly two places — the grokking transition (where it forms, locks, and briefly disorganizes at ~4000) and the ~27k event (a small, reversible perturbation). Everywhere else the representation is frozen and the *weights* are in motion, rotating at equilibrium in the function's null directions. "Weights move, representation holds" is not a slogan here but two measurements on one axis: flat CKA over a window where the weights demonstrably keep rotating.

**Open threads:**

- **p113 control.** The clean grokker should show the same early-formation + plateau-rotational-equilibrium profile but *no* 27k radial escape — the test that the escape (not the rotation) is the event.
- **Which directions is the plateau rotation in?** Project the tangential motion onto the frequency/gauge basis (notebook 1's Procrustes frame) — is the rotation confined to the operand-embedding gauge (the §5 Procrustes thread) or spread across sites?
- **Does the ~4000 disorganization have a weight signature?** Pair the CKA dip with the angular-update spike at grokking (median 1.96°/step in 2–5k, ~10× the plateau) — is the representation churn the downstream face of a rotation burst?
- **Promote.** `representation_similarity` (CKA-to-reference per site) and `rotational_dynamics` (angular update + radial/tangential + equilibrium-norm CV) are the two analyzer candidates this notebook adds (see `docs/notes/analysis_ideas.md`).